In [1]:
import numpy as np
from numba import jit, prange, njit

In [2]:
n_elem_rough = 4

z_surf = np.loadtxt(fname="rough_surface_0.dat",delimiter=";",usecols=range(n_elem_rough+1)) 
z_surf = z_surf.reshape((n_elem_rough+1)*(n_elem_rough+1),1)/500.0

In [3]:
def rough_block(z_surf, nx, ny, nz): 

    def _get_node_index(i_x, i_y, i_z):
                return (nz + 1) * (ny + 1) * i_x + (nz + 1) * i_y + i_z

    #def _get_element_index(i_x, i_y, i_z):
    #    return nz * ny * i_x + nz * i_y + i_z

    x_range = np.linspace(0.,1.0,nx+1)
    y_range = np.linspace(0.,1.0,ny+1) 
    # Create the nodes.
    coordinates = np.zeros(((nx + 1) * (ny + 1) * (nz + 1), 3))
    for i_x in range(nx+1):
        x = x_range[i_x]
        for i_y in range(ny+1):
            y = y_range[i_y]
            z_top = z_surf[i_x*(nx+1)+i_y] + 1.0
            for i_z in range(nz+1):
                z = i_z * z_top / nz
                if (z < 0):
                    raise ValueError('The function should be positive in the '
                        'whole block.')
                coordinates[_get_node_index(i_x, i_y, i_z), :] = [x, y, z[0]]

In [4]:
%%timeit
rough_block(z_surf, nx=n_elem_rough,ny=n_elem_rough, nz=n_elem_rough)

553 µs ± 9.12 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [5]:
@jit
def go_rough_block(z_surf, nx, ny, nz): 

    def _get_node_index(i_x, i_y, i_z):
        return (nz + 1) * (ny + 1) * i_x + (nz + 1) * i_y + i_z

    #def _get_element_index(i_x, i_y, i_z):
    #    return nz * ny * i_x + nz * i_y + i_z

    x_range = np.linspace(0.,1.0,nx+1)
    y_range = np.linspace(0.,1.0,ny+1) 
    # Create the nodes.
    coordinates = np.zeros(((nx + 1) * (ny + 1) * (nz + 1), 3))
    for i_x in range(nx+1):
        x = x_range[i_x]
        for i_y in range(ny+1):
            y = y_range[i_y]
            z_top = z_surf[i_x*(nx+1)+i_y] + 1.0
            for i_z in range(nz+1):
                z = i_z * z_top / nz
                if (z < 0):
                    raise ValueError('The function should be positive in the '
                        'whole block.')
                coordinates[_get_node_index(i_x, i_y, i_z), :] = [x, y, z[0]]
                #coordinates[(nz + 1) * (ny + 1) * i_x + (nz + 1) * i_y + i_z, :] = [x, y, z[0]

In [6]:
%%timeit
go_rough_block(z_surf, nx=n_elem_rough,ny=n_elem_rough, nz=n_elem_rough)

40.1 µs ± 14.2 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%%timeit
go_rough_block(z_surf, nx=n_elem_rough,ny=n_elem_rough, nz=n_elem_rough)

29.9 µs ± 259 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [7]:
@njit(fastmath=True)
def para_rough_block(z_surf, nx, ny, nz): 

    def _get_node_index(i_x, i_y, i_z):
        return (nz + 1) * (ny + 1) * i_x + (nz + 1) * i_y + i_z

    #def _get_element_index(i_x, i_y, i_z):
    #    return nz * ny * i_x + nz * i_y + i_z

    x_range = np.linspace(0.,1.0,nx+1)
    y_range = np.linspace(0.,1.0,ny+1) 
    # Create the nodes.
    coordinates = np.zeros(((nx + 1) * (ny + 1) * (nz + 1), 3))
    for i_x in prange(nx+1):
        x = x_range[i_x]
        for i_y in prange(ny+1):
            y = y_range[i_y]
            z_top = z_surf[i_x*(nx+1)+i_y] + 1.0
            for i_z in prange(nz+1):
                z = i_z * z_top / nz
                if (z < 0):
                    raise ValueError('The function should be positive in the '
                        'whole block.')
                coordinates[_get_node_index(i_x, i_y, i_z), :] = [x, y, z[0]]
                #coordinates[(nz + 1) * (ny + 1) * i_x + (nz + 1) * i_y + i_z, :] = [x, y, z[0]]

In [8]:
%%timeit
para_rough_block(z_surf, nx=n_elem_rough,ny=n_elem_rough, nz=n_elem_rough)

34.7 µs ± 9.1 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit
para_rough_block(z_surf, nx=n_elem_rough,ny=n_elem_rough, nz=n_elem_rough)

29.4 µs ± 192 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)
